# VLM grass classifier (prototype)

Shoulder grass height → Motiva bands: `baixa` | `média` | `alta` (or `null`).

- Model: `gemma-4-26b-a4b-it` (`VLM_MODEL` overrides)
- Live: `GOOGLE_API_KEY` in `services/ai/.env` or the environment
- Fake: `VLM_FAKE=1` (filename stub; no API call)
- Scope: faixa junto à pista only, not distant vegetation
- Bands: <10 cm baixa; 10-30 cm média; >30 cm alta; else null

Kernel: **Python (verdia-ai)** (`services/ai/.venv`).


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

from IPython.display import Image, display
from PIL import Image as PILImage

cwd = Path.cwd().resolve()
for candidate in (cwd, *cwd.parents):
    if (candidate / "pyproject.toml").is_file() and (
        candidate / "src" / "verdia_ai"
    ).is_dir():
        AI_ROOT = candidate
        break
else:
    raise RuntimeError(
        "Could not find services/ai. "
        "Use kernel Python (verdia-ai), or: "
        "cd services/ai && uv run jupyter notebook notebooks/demo_vlm_grass.ipynb"
    )

src = AI_ROOT / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

SAMPLES = AI_ROOT / "notebooks" / "samples"
env_path = AI_ROOT / ".env"
if env_path.is_file():
    for raw in env_path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        key = key.strip()
        value = value.strip().strip("'\"")
        if key and key not in os.environ:
            os.environ[key] = value

from verdia_ai.vlm import (
    DEFAULT_MODEL,
    IMAGE_SUFFIXES,
    classify_image,
    resolve_model,
    use_fake_mode,
)

key_present = bool((os.environ.get("GOOGLE_API_KEY") or "").strip())
fake = use_fake_mode()
model = resolve_model()
print(
    f"{AI_ROOT} | {'FAKE' if fake else 'LIVE'} | {model} | "
    f"key={'yes' if key_present else 'no'} | .env={'yes' if env_path.is_file() else 'no'}"
)


## Image

Paste paths into `IMAGES`, or `DEMO=True` for `SAMPLES`.


In [ ]:
DEMO = False
IMAGES = [
    Path("path/to/image1.jpg"),
    Path("path/to/image2.jpg"),
]

if DEMO:
    paths = sorted(
        p for p in SAMPLES.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES
    )
    assert paths, f"no images in {SAMPLES}"
else:
    paths = [p.expanduser().resolve() for p in IMAGES]
    missing = [str(p) for p in paths if not p.is_file()]
    assert not missing, f"IMAGE not found: {missing}"

for path in paths:
    im = PILImage.open(path)
    print(f"{path.name} - {im.size[0]}x{im.size[1]} (faixa junto à pista)")
    display(Image(filename=str(path), width=420))


## Classify

Fields: `classe`, `altura_estimada_cm`, `confianca_declarada`, `justificativa`.


In [ ]:
def pretty(verdict_dict: dict) -> str:
    return json.dumps(verdict_dict, ensure_ascii=False, indent=2)


if fake:
    print("VLM_FAKE=1: stub only.")
elif not key_present:
    raise SystemExit(
        "GOOGLE_API_KEY required for live calls "
        "(services/ai/.env or VLM_FAKE=1)."
    )

results = []
for path in paths:
    print("=" * 60)
    print(path.name)
    display(Image(filename=str(path), width=280))
    verdict = classify_image(path)
    row = verdict.to_dict()
    row["path"] = path.name
    results.append(row)
    print(pretty(row))

print("=" * 60)
mode = "fake" if results and results[0].get("fake") else "live"
print(f"done: {len(results)} image(s), mode={mode}")
